In [0]:
# Databricks notebook source

# MAGIC %run /Users/sandysakthivel2005@gmail.com/common/03_Logger

from pyspark.sql.functions import *

try:

    logger.info("Silver Sentiment Pipeline Started")
    print("Silver Sentiment Pipeline Started")

    # ==========================================
    # Read Bronze Streaming Table
    # ==========================================

    bronzeDF = (
        spark.readStream
             .table("bronze_catalog1.raw.bronze_sentiment1")
    )

    logger.info("Bronze Streaming Table Read Successfully")
    print("Bronze Streaming Table Read Successfully")

    # ==========================================
    # Remove Duplicates
    # ==========================================

    silverDF = bronzeDF.dropDuplicates(["tweet_id"])

    # ==========================================
    # Handle Null Values
    # ==========================================

    silverDF = (
        silverDF
            .fillna({
                "sentiment_score": 0,
                "positive_score": 0,
                "negative_score": 0,
                "neutral_score": 0,
                "impressions": 0,
                "likes": 0,
                "engagement_count": 0,
                "topic_category": "Unknown"
            })
    )

    # ==========================================
    # Data Validation
    # ==========================================

    silverDF = (
        silverDF
            .filter(col("tweet_id").isNotNull())
            .filter(col("topic_category").isNotNull())
            .filter(col("tweet_timestamp").isNotNull())
            .filter(col("sentiment_score") >= -1)
            .filter(col("sentiment_score") <= 1)
            .filter(col("positive_score") >= 0)
            .filter(col("negative_score") >= 0)
            .filter(col("neutral_score") >= 0)
            .filter(col("impressions") >= 0)
            .filter(col("likes") >= 0)
            .filter(col("engagement_count") >= 0)
    )

    # ==========================================
    # Standardize Text
    # ==========================================

    silverDF = (
        silverDF
            .withColumn("topic_category", upper(trim(col("topic_category"))))
    )

    # ==========================================
    # Convert Data Types
    # ==========================================

    silverDF = (
        silverDF
            .withColumn("tweet_timestamp", to_timestamp(col("tweet_timestamp")))
            .withColumn("sentiment_score", col("sentiment_score").cast("double"))
            .withColumn("positive_score", col("positive_score").cast("double"))
            .withColumn("negative_score", col("negative_score").cast("double"))
            .withColumn("neutral_score", col("neutral_score").cast("double"))
            .withColumn("impressions", col("impressions").cast("int"))
            .withColumn("likes", col("likes").cast("int"))
            .withColumn("engagement_count", col("engagement_count").cast("int"))
    )

    # ==========================================
    # Standardize Date
    # ==========================================

    silverDF = (
        silverDF
            .withColumn("tweet_date", to_date(col("tweet_timestamp")))
    )

    # ==========================================
    # Audit Columns
    # ==========================================

    silverDF = (
        silverDF
            .withColumn("silver_load_time", current_timestamp())
            .withColumn("pipeline_name", lit("Silver_Sentiment"))
    )

    logger.info("Silver Transformations Completed Successfully")
    print("Silver Transformations Completed Successfully")

    # ==========================================
    # Write Silver Table
    # ==========================================

    silverQuery = (
        silverDF.writeStream
            .trigger(availableNow=True)
            .format("delta")
            .outputMode("append")
            .option(
                "checkpointLocation",
                "abfss://socialmedia@socialmediaadls001.dfs.core.windows.net/checkpoints/silver_sentiment"
            )
            .option("mergeSchema", "true")
            .toTable("silver_catalog1.processed.silver_sentiment")
    )

    silverQuery.awaitTermination()

    logger.info("Silver Sentiment Loaded Successfully")
    print("Silver Sentiment Loaded Successfully")

except Exception as e:

    logger.error(f"Silver Sentiment Pipeline Failed: {str(e)}")
    print(f"Silver Sentiment Pipeline Failed: {str(e)}")
    raise

Silver Sentiment Pipeline Started
Bronze Streaming Table Read Successfully
Silver Transformations Completed Successfully
Silver Sentiment Loaded Successfully


In [0]:
%run /Users/sandysakthivel2005@gmail.com/common/03_Logger

In [0]:
print(logger)

<Logger SocialMediaPipeline (WARNING)>


Logger notebook executed successfully


In [0]:
dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()

'/Users/sandysakthivel2005@gmail.com/02_Silver_Sentiment'

In [0]:
%sql
SELECT COUNT(*)
FROM silver_catalog1.processed.silver_sentiment;

count(1)
2638


In [0]:
%sql
SELECT *
FROM silver_catalog1.processed.silver_sentiment
LIMIT 20;

tweet_id,topic_category,tweet_timestamp,sentiment_score,positive_score,negative_score,neutral_score,impressions,likes,engagement_count,bronze_load_time,pipeline_name,source_system,ingestion_date,tweet_date,silver_load_time
T12824,CLOUD,2025-01-20T16:10:00Z,0.70076387,0.687240832,0.143045822,0.159310371,1173,3941,763,2026-07-09T09:17:28.261Z,Silver_Sentiment,Azure Event Hub,2026-07-09,2025-01-20,2026-07-10T04:17:12.256Z
T16893,CLOUD,2025-01-05T12:29:00Z,-0.716256012,0.727864661,0.431720768,0.0,613,1160,800,2026-07-09T09:17:28.261Z,Silver_Sentiment,Azure Event Hub,2026-07-09,2025-01-05,2026-07-10T04:17:12.256Z
T17893,FINANCE,2025-01-11T04:24:00Z,-0.403860089,0.499100257,0.006239251,0.0,0,327,782,2026-07-09T09:17:28.261Z,Silver_Sentiment,Azure Event Hub,2026-07-09,2025-01-11,2026-07-10T04:17:12.256Z
T38549,FINANCE,2025-01-21T08:04:00Z,-0.704903966,0.640689225,0.685762878,0.145752289,9521,764,838,2026-07-09T09:17:28.261Z,Silver_Sentiment,Azure Event Hub,2026-07-09,2025-01-21,2026-07-10T04:17:12.256Z
T28605,AI,2025-01-16T01:13:00Z,-0.654200491,0.254571857,0.789469493,0.771584128,1604,1164,873,2026-07-09T09:17:28.261Z,Silver_Sentiment,Azure Event Hub,2026-07-09,2025-01-16,2026-07-10T04:17:12.256Z
T26303,FINANCE,2025-01-02T12:33:00Z,-0.192737498,0.509779332,0.371650405,0.277859995,5716,4854,258,2026-07-09T09:17:28.261Z,Silver_Sentiment,Azure Event Hub,2026-07-09,2025-01-02,2026-07-10T04:17:12.256Z
T39869,SPORTS,2025-01-01T20:17:00Z,-0.617944653,0.170905399,0.157232324,0.08172675,19979,3888,160,2026-07-09T09:17:28.261Z,Silver_Sentiment,Azure Event Hub,2026-07-09,2025-01-01,2026-07-10T04:17:12.256Z
T36618,SPORTS,2025-01-17T12:11:00Z,-0.673454751,0.917374923,0.295530621,0.216306098,12029,548,965,2026-07-09T09:17:28.261Z,Silver_Sentiment,Azure Event Hub,2026-07-09,2025-01-17,2026-07-10T04:17:12.256Z
T4720,FINANCE,2025-01-03T02:39:00Z,0.517499736,0.046797007,0.404950821,0.280693634,19960,79,407,2026-07-09T09:17:28.261Z,Silver_Sentiment,Azure Event Hub,2026-07-09,2025-01-03,2026-07-10T04:17:12.256Z
T16540,SPORTS,2025-01-10T08:25:00Z,-0.462649924,0.288138057,0.12594547,0.306230064,0,4734,71,2026-07-09T09:17:28.261Z,Silver_Sentiment,Azure Event Hub,2026-07-09,2025-01-10,2026-07-10T04:17:12.256Z


In [0]:
%sql
SELECT
MIN(sentiment_score) AS min_score,
MAX(sentiment_score) AS max_score
FROM silver_catalog1.processed.silver_sentiment;

min_score,max_score
-0.999239358,0.999076847
